# STA 556 Activity Reference Notebook

**Statistics and Data Science Computing Workflows**

This notebook follows the edited activity used in class. Python examples are executable cells; terminal, SQL, YAML, configuration, and text examples remain displayed as Markdown where appropriate.

**Reference answers are placed immediately after questions, challenges, predictions, and reflections.** They show one reasonable solution or interpretation; other correct approaches may exist.

# 2.2 Activity

## Joining, Reshaping & Missing Data

**Tools:** Python, pandas, VS Code and/or Jupyter

## Learning objectives

By the end of this tutorial, you should be able to:

- combine tables using `merge()`;
- distinguish inner, left, right, and outer joins;
- identify and diagnose unmatched records;
- detect duplicate keys and validate join cardinality;
- combine datasets using `concat()`;
- convert data between wide and long form;
- use `melt()`, `pivot()`, and `pivot_table()`;
- detect and summarize missing data;
- apply simple missing-data strategies critically;
- validate a multi-step wrangling pipeline.

---

# Part 0 — Set up

Create:

```text
notebooks/week04_wrangle_part2.ipynb
```

At the top:

In [ ]:
import pandas as pd
import numpy as np

---

# Part 1 — Create two related tables

Create participant information:

In [ ]:
participants = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106],
    "age": [24, 31, 27, 19, 42, 35],
    "group": ["A", "B", "A", "B", "A", "B"]
})

Create scores:

In [ ]:
scores = pd.DataFrame({
    "id": [101, 102, 103, 105, 106, 107],
    "score": [81, 94, 88, 91, 85, 79]
})

Inspect:

In [ ]:
participants
scores

### Questions

1. Which IDs occur in both tables?
2. Which ID occurs only in `participants`?
3. Which ID occurs only in `scores`?
4. What do you expect an inner join to contain?

Do not run a merge yet.

---

# Part 2 — Inner join

Run:

In [ ]:
inner = participants.merge(
    scores,
    on="id",
    how="inner"
)

Inspect:

In [ ]:
inner
inner.shape

### Reference answer

Shared IDs: `101, 102, 103, 105, 106`; `104` is only in participants; `107` is only in scores. An inner join keeps the five shared IDs.

### Questions

1. Which IDs were removed?
2. Why?
3. Did the number of rows match your prediction?

---

# Part 3 — Left join

Run:

In [ ]:
left = participants.merge(
    scores,
    on="id",
    how="left"
)

Inspect:

In [ ]:
left

### Reference answer

`104` and `107` are removed by the inner join because they are unmatched.

### Questions

1. Which table defines the resulting population?
2. What happened to ID 104?
3. Why is its score missing?

---

# Part 4 — Outer join

Run:

In [ ]:
outer = participants.merge(
    scores,
    on="id",
    how="outer"
)

Inspect:

In [ ]:
outer

### Reference answer

The left table defines the retained population. ID `104` remains with a missing score.

### Challenge

Before running a right join, predict which IDs it will contain.

Then:

In [ ]:
right = participants.merge(
    scores,
    on="id",
    how="right"
)

### Reference answer

The right join keeps all IDs in `scores`: `101, 102, 103, 105, 106, 107`.

---

# Part 5 — Diagnose joins with `indicator=True`

Run:

In [ ]:
diagnostic = participants.merge(
    scores,
    on="id",
    how="outer",
    indicator=True
)

Inspect:

In [ ]:
diagnostic

Then:

In [ ]:
diagnostic["_merge"].value_counts()

### Questions

1. How many rows matched?
2. How many occurred only in the participant table?
3. How many occurred only in the score table?

### Challenge

Extract only unmatched records.

---

# Part 6 — Check key uniqueness

Run:

In [ ]:
participants["id"].is_unique
scores["id"].is_unique

Then:

In [ ]:
participants["id"].duplicated().sum()
scores["id"].duplicated().sum()

### Reference answer

Expected: 5 `both`, 1 `left_only`, and 1 `right_only`.

### Question

Why should this be checked before a one-to-one join?

---

# Part 7 — Validate the merge

Run:

In [ ]:
validated = participants.merge(
    scores,
    on="id",
    how="left",
    validate="one_to_one"
)

Now create a bad score table:

In [ ]:
duplicate_scores = pd.DataFrame({
    "id": [101, 101, 102],
    "score": [81, 83, 94]
})

Try:

In [ ]:
participants.merge(
    duplicate_scores,
    on="id",
    validate="one_to_one"
)

### Question

Why is the error valuable?

---

# Part 8 — Many-to-many multiplication

Create:

In [ ]:
visits = pd.DataFrame({
    "id": [1, 1, 2],
    "visit": ["pre", "post", "pre"]
})

treatments = pd.DataFrame({
    "id": [1, 1, 2],
    "treatment": ["A", "B", "A"]
})

Run:

In [ ]:
visits.merge(
    treatments,
    on="id"
)

### Questions

1. Why does ID 1 generate four rows?
2. Was information duplicated?
3. Under what circumstances could this be correct?

---

# Part 9 — Concatenate rows

Create two semester datasets:

In [ ]:
fall = pd.DataFrame({
    "id": [1, 2, 3],
    "semester": ["Fall"] * 3,
    "score": [80, 85, 90]
})

spring = pd.DataFrame({
    "id": [4, 5, 6],
    "semester": ["Spring"] * 3,
    "score": [78, 92, 88]
})

Combine:

In [ ]:
all_scores = pd.concat(
    [fall, spring],
    axis=0,
    ignore_index=True
)

Inspect:

In [ ]:
all_scores

### Question

Why is `concat()` more natural than `merge()` here?

---

# Part 10 — Concatenating mismatched columns

Create:

In [ ]:
fall = pd.DataFrame({
    "id": [1, 2],
    "score": [80, 85]
})

spring = pd.DataFrame({
    "id": [3, 4],
    "score": [90, 92],
    "campus": ["Flagstaff", "Flagstaff"]
})

Run:

In [ ]:
pd.concat(
    [fall, spring],
    ignore_index=True
)

### Question

What does pandas do with the column that does not exist in `fall`?

---

# Part 11 — Wide data

Create repeated measurements:

In [ ]:
wide = pd.DataFrame({
    "id": [101, 102, 103, 104],
    "group": ["A", "B", "A", "B"],
    "score_pre": [72, 85, 79, 88],
    "score_post": [81, 91, 84, 90]
})

Inspect:

In [ ]:
wide

### Question

How many rows represent each participant?

How are repeated measurements represented?

---

# Part 12 — Wide to long with `melt()`

Run:

In [ ]:
long = wide.melt(
    id_vars=["id", "group"],
    value_vars=["score_pre", "score_post"],
    var_name="time",
    value_name="score"
)

Inspect:

In [ ]:
long

### Reference answer

Wide data store repeated measures across columns; long data store them across rows. Use `melt()` for wide→long and `pivot()` for long→wide when combinations are unique.

### Questions

1. How many rows now represent each participant?
2. What happened to the original score columns?
3. What does `id_vars` mean?
4. What does `value_vars` mean?

---

# Part 13 — Clean the melted variable

Run:

In [ ]:
long["time"] = (
    long["time"]
    .str.replace("score_", "", regex=False)
)

Inspect:

In [ ]:
long

### Challenge

Change:

```text
pre
post
```

to:

```text
baseline
followup
```

using:

In [ ]:
.replace()

---

# Part 14 — Long to wide with `pivot()`

Start from `long`.

Run:

In [ ]:
wide_again = long.pivot(
    index=["id", "group"],
    columns="time",
    values="score"
)

Inspect:

In [ ]:
wide_again

Reset the index:

In [ ]:
wide_again = wide_again.reset_index()

### Question

Did you recover the same information as the original wide dataset?

---

# Part 15 — Duplicate combinations and `pivot()`

Create:

In [ ]:
repeated = pd.DataFrame({
    "id": [1, 1, 1, 2, 2],
    "time": ["pre", "pre", "post", "pre", "post"],
    "score": [70, 74, 82, 80, 88]
})

Try:

In [ ]:
repeated.pivot(
    index="id",
    columns="time",
    values="score"
)

### Question

Why does this fail?

Which assumption is violated?

---

# Part 16 — `pivot_table()`

Now use:

In [ ]:
summary = repeated.pivot_table(
    index="id",
    columns="time",
    values="score",
    aggfunc="mean"
)

Inspect:

In [ ]:
summary

### Reflection

Why should you investigate duplicate records before deciding that taking the mean is appropriate?

---

# Part 17 — Create missing data

Create:

In [ ]:
clinical = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "age": [24, 31, np.nan, 45, 28, 39],
    "group": ["A", "B", "A", "B", None, "A"],
    "score": [81, np.nan, 77, 92, 88, np.nan]
})

Inspect:

In [ ]:
clinical

Then:

In [ ]:
clinical.isna()

and:

In [ ]:
clinical.isna().sum()

Calculate percentages:

In [ ]:
clinical.isna().mean() * 100

---

# Part 18 — Identify rows with missing data

Find rows with missing score:

In [ ]:
clinical.loc[
    clinical["score"].isna()
]

Find complete scores:

In [ ]:
clinical.loc[
    clinical["score"].notna()
]

Find rows where either age OR score is missing:

In [ ]:
clinical.loc[
    clinical[["age", "score"]]
    .isna()
    .any(axis=1)
]

### Challenge

Find rows where both age and score are observed.

---

# Part 19 — Drop missing observations

Run:

In [ ]:
complete_score = clinical.dropna(
    subset=["score"]
)

Compare:

In [ ]:
len(clinical)
len(complete_score)

Now require both age and score:

In [ ]:
complete_cases = clinical.dropna(
    subset=["age", "score"]
)

### Questions

1. How many observations were removed?
2. Has the analysis population changed?
3. What assumptions might make complete-case analysis problematic?

---

# Part 20 — Filling missing values

Create a copy:

In [ ]:
filled = clinical.copy()

Fill scores with the mean:

In [ ]:
filled["score"] = filled["score"].fillna(
    filled["score"].mean()
)

Compare:

In [ ]:
clinical["score"].describe()
filled["score"].describe()

### Questions

1. Did the mean change?
2. Did the standard deviation change?
3. Why?
4. Is mean imputation automatically a good statistical strategy?

---

# Part 21 — Missingness introduced by a join

Return to:

In [ ]:
participants
scores

Create:

In [ ]:
joined = participants.merge(
    scores,
    on="id",
    how="left"
)

Find:

In [ ]:
joined.loc[
    joined["score"].isna()
]

### Question

Does a missing score here necessarily mean that the participant failed to provide a score?

What other explanation exists?

---

# Part 22 — Missingness indicator

Create:

In [ ]:
clinical["score_missing"] = (
    clinical["score"].isna()
)

Inspect:

In [ ]:
clinical

Now:

In [ ]:
pd.crosstab(
    clinical["group"],
    clinical["score_missing"]
)

### Reflection

What question does this table help you investigate?

---

# Part 23 — Build a complete wrangling workflow

Create demographic data:

In [ ]:
demographics = pd.DataFrame({
    "id": [101, 102, 103, 104, 105],
    "age": [24, 31, 27, 42, 36],
    "group": ["A", "B", "A", "B", "A"]
})

Create repeated outcomes:

In [ ]:
outcomes = pd.DataFrame({
    "id": [101, 102, 103, 105],
    "pre": [72, 80, 85, 78],
    "post": [81, 87, np.nan, 86]
})

Your task is to produce a long-format analysis dataset containing:

```text
id
age
group
time
score
```

with all demographic participants preserved.

---

# Part 24 — Step 1: audit the keys

Check:

In [ ]:
demographics["id"].is_unique
outcomes["id"].is_unique

Compare sets of IDs:

In [ ]:
set(demographics["id"]) - set(outcomes["id"])

and:

In [ ]:
set(outcomes["id"]) - set(demographics["id"])

Write down what you expect the merge to do.

---

# Part 25 — Step 2: merge

Use:

In [ ]:
combined = demographics.merge(
    outcomes,
    on="id",
    how="left",
    validate="one_to_one",
    indicator=True
)

Inspect:

In [ ]:
combined
combined["_merge"].value_counts()

Remove the diagnostic column after checking:

In [ ]:
combined = combined.drop(
    columns="_merge"
)

---

# Part 26 — Step 3: reshape

Convert to long format:

In [ ]:
analysis = combined.melt(
    id_vars=["id", "age", "group"],
    value_vars=["pre", "post"],
    var_name="time",
    value_name="score"
)

Inspect:

In [ ]:
analysis

---

# Part 27 — Step 4: audit missingness

Run:

In [ ]:
analysis.isna().sum()

Then:

In [ ]:
analysis.loc[
    analysis["score"].isna()
]

### Questions

For each missing score, determine whether it arose because:

1. an outcome record was absent entirely; or
2. an outcome row existed but a particular measurement was missing.

These are different data problems.

---

# Part 28 — Step 5: define an analysis dataset

Suppose the analysis requires observed scores.

Create:

In [ ]:
observed = analysis.dropna(
    subset=["score"]
)

Record:

In [ ]:
n_before = len(analysis)
n_after = len(observed)
n_removed = n_before - n_after

Print these quantities.

### Reflection

Why should this information appear in an analysis report?

---

# Part 29 — Validate the transformation

Check:

In [ ]:
analysis["id"].nunique()

Compare with:

In [ ]:
demographics["id"].nunique()

Check time values:

In [ ]:
analysis["time"].value_counts()

Check participant-time uniqueness:

In [ ]:
analysis.duplicated(
    subset=["id", "time"]
).sum()

Add assertions:

In [ ]:
assert (
    analysis["id"].nunique()
    == demographics["id"].nunique()
)

assert (
    analysis.duplicated(
        subset=["id", "time"]
    ).sum()
    == 0
)

---

# Part 30 — Mini challenge: three files

Imagine three monthly extracts:

In [ ]:
jan = pd.DataFrame({
    "id": [1, 2],
    "month": ["Jan", "Jan"],
    "score": [80, 85]
})

feb = pd.DataFrame({
    "id": [1, 2],
    "month": ["Feb", "Feb"],
    "score": [82, 88]
})

mar = pd.DataFrame({
    "id": [1, 2],
    "month": ["Mar", "Mar"],
    "score": [84, np.nan]
})

### Tasks

1. Concatenate the three tables.
2. Verify the number of rows.
3. Identify missing scores.
4. Pivot into wide form with one row per participant.
5. Convert back to long form.
6. Verify that the structure is equivalent to the original combined data.

---

# Part 31 — Written reasoning challenge

For each scenario, choose the most appropriate operation.

### A

Two tables contain the same variables for different months.

```text
merge or concat?
```

### B

One table contains demographics and another contains outcomes linked by participant ID.

```text
merge or concat?
```

### C

Repeated measurements are stored in `score_pre`, `score_post`, and `score_followup`.

```text
melt or pivot?
```

### D

Long data should become one row per participant.

```text
melt or pivot?
```

### E

You need every participant from the enrollment table, whether or not an outcome exists.

```text
inner or left join?
```

Explain each answer.

---

# Part 32 — Git checkpoint

Run:

```bash
git status
```

Stage:

```bash
git add .
```

Commit:

```bash
git commit -m "Complete Week 4 joining reshaping and missing data"
```

Push:

```bash
git push
```

---

# Part 33 — Final reflection

Answer in Markdown.

### 1. Merge

What is the conceptual difference between an inner join and a left join?

### 2. Keys

Why should you inspect duplicate keys before merging?

### 3. Validation

What does `validate="one_to_one"` protect against?

### 4. Concatenation

When is `concat()` more appropriate than `merge()`?

### 5. Reshaping

What is the difference between wide and long data?

### 6. Pivoting

Why might `pivot()` fail when `pivot_table()` succeeds?

### 7. Missingness

Why should missing values introduced by a join be interpreted differently from missing values already present in a source table?

### 8. Statistical practice

Why is dropping missing observations a statistical decision rather than only a programming operation?

---

# Completion checklist

- [ ] Created Week 4 notebook
- [ ] Performed an inner join
- [ ] Performed a left join
- [ ] Performed a right join
- [ ] Performed an outer join
- [ ] Used `indicator=True`
- [ ] Checked duplicate keys
- [ ] Used `validate=`
- [ ] Explored a many-to-many merge
- [ ] Concatenated rows
- [ ] Examined mismatched columns during concatenation
- [ ] Created wide-format data
- [ ] Used `melt()`
- [ ] Used `pivot()`
- [ ] Used `pivot_table()`
- [ ] Detected missing values
- [ ] Calculated missing-data percentages
- [ ] Used `dropna()`
- [ ] Used `fillna()`
- [ ] Compared distributions before and after filling
- [ ] Created a missingness indicator
- [ ] Completed the full merge → reshape → missingness workflow
- [ ] Added validation assertions
- [ ] Completed the monthly-data challenge
- [ ] Committed work to Git
- [ ] Pushed work to GitHub

---

# What you should now understand

```text
Separate datasets
       ↓
Identify keys
       ↓
Check relationships
       ↓
Merge / concatenate
       ↓
Audit
       ↓
Reshape
       ↓
Inspect missingness
       ↓
Make statistical decisions
       ↓
Validate
       ↓
Analysis-ready data
```

Next week we will work with **external data sources**, including SQL databases, APIs, JSON, HTML, and binary file formats.